# Dataset processing

Input data: 
- ParlaMint dataset (handle), where the data is merged together into a master preprocessed dataset, to be used for all of the analysis. Specifically, we include the following data:

Sample dataset (for replicability purposes:) in the [Sample](../Sample/ParlaMint-SI5.1_Sample/):
- [Sample.txt](../Sample/ParlaMint-SI5.1_Sample/Sample.txt/) (text files, to acquire the texts)
- [Sample.conllu](../Sample/ParlaMint-SI5.1_Sample/Sample.conllu/) (metadata and sentiment metadata files)
- [CHES dataset for Slovenian data](../Sample/CHES-SI.tsv)

Sample timeframe:
This sample covers Nov 2019 – Apr 2020, a key period in Slovenian politics that overlaps with the start of COVID-19. It’s included both to demonstrate the analysis workflow and to give a snapshot of party and speech trends during this timeframe—useful context when interpreting results from the full dataset.


Full dataset available for download on the CLARIN.SI repository: 
> Erjavec, Tomaž; et al., 2025, **Multilingual comparable corpora of parliamentary debates ParlaMint 5.0**, Slovenian language resource repository CLARIN.SI, ISSN 2820-4042, 
http://hdl.handle.net/11356/2004.

Full CHES dataset is available for download through Chapel Hill Expert Survey website:
> Jolly, Seth, Ryan Bakker, Liesbet Hooghe, Gary Marks, Jonathan Polk, Jan Rovny, Marco Steenbergen, and Milada Anna Vachudova. 2022. **Chapel Hill Expert Survey Trend File, 1999-2019.** Electoral Studies 75 (February). https://doi.org/10.1016/j.electstud.2021.102420

The already processed CHES-SI dataset (with aligned ParlaMint party IDs) is available through [ParlaMint GitHub site](https://github.com/clarin-eric/ParlaMint/tree/main/Build/Sources-TSV/ParlaMint-SI/OrientationCHES-SI.edited.tsv).


Output:
- Pre-processed and merged master dataset [ParlaMint-SI_5.1_master.tsv](../Sample/Datasets/Sample_master.tsv)

## Preprocessing ParlaMint-SI corpus

In [1]:
import pandas as pd
import os
import glob

In [2]:
#Sample datasets, replace with actual ParlaMint dataset from CLARIN.SI repository entry
text_dir = "../Sample/ParlaMint-SI5.1_Sample/Sample.txt"
conllu_dir = "../Sample/ParlaMint-SI5.1_Sample/Sample.conllu"


In [3]:
# Text files to extrac ID and text
text_files = glob.glob(os.path.join(text_dir, '*', '*.txt'))

#Session metadata
text_meta = glob.glob(os.path.join(text_dir, '*', '*-meta-en.tsv'))

#Sentiment-related metadata
sent_files = glob.glob(os.path.join(conllu_dir, '**', '*-ana-meta-en.tsv'))

print(text_meta)

['../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-11-21-SDZ8-Redna-13-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-12-20-SDZ8-Redna-14-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-11-26-SDZ8-Redna-13-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-11-18-SDZ8-Redna-13-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-12-16-SDZ8-Redna-14-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-11-22-SDZ8-Izredna-28-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-12-19-SDZ8-Redna-14-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-12-18-SDZ8-Redna-14-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-11-22-SDZ8-Redna-13-meta-en.tsv', '../Sample/ParlaMint-SI5.1_Sample/Sample.txt/2019/ParlaMint-SI_2019-11-29-SDZ8-

In [4]:
id = []
texts = []
#Processing text files
for file in text_files:
    with open(file, 'r', encoding="utf-8") as infile:
        for line in infile:
            if '\t' in line:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    id.append(parts[0])
                    texts.append(parts[1])


df_texts = pd.DataFrame({'ID': id, 'Text':texts})
print("df_meta shape: ", df_texts.shape)

df_texts.head()


df_meta shape:  (5404, 2)


,ID,Text
0,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u1,"Spoštovane poslanke, spoštovani poslanci, gosp..."
1,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u2,"Spoštovani predsednik Državnega zbora, hvala z..."
2,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u3,"Hvala, gospa direktorica, za dopolnilno obrazl..."
3,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u4,"Spoštovani predsedujoči, kolegice in kolegi! O..."
4,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u5,"Hvala, gospod predsednik, za predstavitev stal..."


In [5]:
df_meta = pd.DataFrame()
for file in text_meta:
    df = pd.read_csv(file, sep='\t', encoding='utf-8')
    df_meta = pd.concat([df_meta, df])

df_meta.head()

cols = ['Text_ID', 'Title', 'Body', 'Session', 'Sitting', 'Agenda', 'Lang']
df_meta = df_meta.drop(columns=cols)

print("df_meta shape: ", df_meta.shape)
df_meta.head()


df_meta shape:  (5404, 17)


,ID,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,Speaker_party_name,Party_status,Party_orientation,Speaker_ID,Speaker_name,Speaker_gender,Speaker_birth,Topic
0,ParlaMint-SI_2019-11-21-SDZ8-Redna-13.u1,2019-11-21,8. mandat,Redna,Reference,Chairperson,MP,notMinister,SDS,Slovenian Democratic Party,Opposition,Right,TankoJože,"Tanko, Jože",M,1956,Macroeconomics
1,ParlaMint-SI_2019-11-21-SDZ8-Redna-13.u2,2019-11-21,8. mandat,Redna,Reference,Regular,MP,notMinister,DeSUS,Democratic Party of Pensioners of Slovenia,Coalition,Centre to centre-left,PolnarRobert,"Polnar, Robert",M,1960,Mix
2,ParlaMint-SI_2019-11-21-SDZ8-Redna-13.u3,2019-11-21,8. mandat,Redna,Reference,Chairperson,MP,notMinister,SDS,Slovenian Democratic Party,Opposition,Right,TankoJože,"Tanko, Jože",M,1956,Macroeconomics
3,ParlaMint-SI_2019-11-21-SDZ8-Redna-13.u4,2019-11-21,8. mandat,Redna,Reference,Regular,notMP,notMinister,-,-,-,-,KovačJerebNatalija,"Kovač Jereb, Natalija",F,-,Macroeconomics
4,ParlaMint-SI_2019-11-21-SDZ8-Redna-13.u5,2019-11-21,8. mandat,Redna,Reference,Chairperson,MP,notMinister,SDS,Slovenian Democratic Party,Opposition,Right,TankoJože,"Tanko, Jože",M,1956,Other


In [6]:
df_sent = pd.DataFrame()
for file in sent_files:
    df = pd.read_csv(file, sep='\t', encoding='utf-8')
    df_sent = pd.concat([df_sent, df])

df_sent = df_sent[df_sent['Element'] == 'u']
cols = ['ID', 'Senti_3', 'Senti_6', 'Senti_n', 'Sents', 'Words', 'Tokens']
df_sent = df_sent[cols].reset_index(drop=True)
df_sent['ID'] = df_sent['ID'].str.replace(".ana", "", regex=False)

print("df_sent shape: ", df_sent.shape)
df_sent.head()


df_sent shape:  (5404, 7)


,ID,Senti_3,Senti_6,Senti_n,Sents,Words,Tokens
0,ParlaMint-SI_2019-11-20-SDZ8-Redna-13.u1,Neutral,neutral positive,3.29,30,690,815
1,ParlaMint-SI_2019-11-20-SDZ8-Redna-13.u2,Negative,mixed negative,0.79,50,776,926
2,ParlaMint-SI_2019-11-20-SDZ8-Redna-13.u3,Neutral,neutral positive,2.97,3,28,35
3,ParlaMint-SI_2019-11-20-SDZ8-Redna-13.u4,Negative,negative,0.08,12,157,201
4,ParlaMint-SI_2019-11-20-SDZ8-Redna-13.u5,Neutral,neutral positive,2.91,38,350,405


In [7]:
parla = pd.merge(pd.merge(df_texts, df_meta, on='ID'), df_sent, on='ID')
# Renaming slv metadata to eng (X. mandat -> Term X)
parla['Term'] = parla['Term'].replace({
    '3. mandat': 'Term 3',
    '4. mandat': 'Term 4',
    '5. mandat': 'Term 5',
    '6. mandat': 'Term 6',
    '7. mandat': 'Term 7',
    '8. mandat': 'Term 8'
})

print("df shape: ", df.shape)
parla.head()

df shape:  (3419, 12)


,ID,Text,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,...,Speaker_name,Speaker_gender,Speaker_birth,Topic,Senti_3,Senti_6,Senti_n,Sents,Words,Tokens
0,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u1,"Spoštovane poslanke, spoštovani poslanci, gosp...",2019-11-22,Term 8,Redna,Reference,Chairperson,MP,notMinister,SD,...,"Židan, Dejan",M,1967,Other,Positive,positive,4.67,26,548,640
1,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u2,"Spoštovani predsednik Državnega zbora, hvala z...",2019-11-22,Term 8,Redna,Reference,Regular,notMP,notMinister,-,...,"Godina, Duška",F,-,Energy,Positive,mixed positive,4.34,25,585,646
2,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u3,"Hvala, gospa direktorica, za dopolnilno obrazl...",2019-11-22,Term 8,Redna,Reference,Chairperson,MP,notMinister,SD,...,"Židan, Dejan",M,1967,Other,Neutral,neutral positive,3.00,3,25,31
3,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u4,"Spoštovani predsedujoči, kolegice in kolegi! O...",2019-11-22,Term 8,Redna,Reference,Regular,MP,notMinister,LMŠ,...,"Paulič, Edvard",M,1968,Energy,Positive,mixed positive,3.51,18,431,467
4,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u5,"Hvala, gospod predsednik, za predstavitev stal...",2019-11-22,Term 8,Redna,Reference,Chairperson,MP,notMinister,SD,...,"Židan, Dejan",M,1967,Other,Neutral,neutral positive,2.97,4,33,44


## CHES preprocessing
Appending CHES data, mapping numerical values

Party abbreaviations in ParlaMint do not match those in CHES (though adapted for ParlaMint, problem with parties such as ZL and Levica, CHES years coincide with changes within the party structure - either rename or just restructure)

In [8]:
ches = pd.read_csv("../Sample/CHES-SI.tsv", sep="\t", encoding="utf-8")
cols = ["parlamint", "year", "lrgen", "galtan", "family", "seat"]
ches = ches[cols]
ches = ches.rename(columns={"parlamint":"Parlamint", "year":"Year", "lrgen":"lrgen", "galtan":"galtan", "family":"Family", "seat":"Seat"})
print(ches.shape)
ches.head()

#Parties in ParlaMint, but not in CHES: DL, DLGV, GAS, Konkretno, Lipa, NP, NeP


(49, 6)


,Parlamint,Year,lrgen,galtan,Family,Seat
0,-,2006,4.6,2.6,7,0
1,-,2010,6.8,7.8,4,5.6
2,DL,-,-,-,-,-
3,DLGV,-,-,-,-,-
4,DeSUS,2002,3.4,5.8,9,4.4


In [9]:
#Mapping families numerical values to string; mapping parties into common (unified) parties column
families={
    "1":"Radical Right", 
    "2":"Conservatives",
    "3":"Liberal", 
    "4":"Christian-Democratic",
    "5":"Socialist", 
    "6":"Radical Left", 
    "7":"Green", 
    "8":"Regionalist", 
    "9":"No family", 
    "10":"Confessional", 
    "11":"Agrarian/Center"
}

parties = {
    "ZL": "ZL/Levica",
    "Levica + ZL":"ZL/Levica",
    "Levica": "ZL/Levica",
    "AB":"ZaAB/ZaSLD/SAB",
    "ZaAB":"ZaAB/ZaSLD/SAB",
    "ZaSLD":"ZaAB/ZaSLD/SAB",
    "SAB" :"ZaAB/ZaSLD/SAB",
    "SAB + ZaSLD":"ZaAB/ZaSLD/SAB",
    "ZLSD":"ZLSD/SD",
    "SD":"ZLSD/SD",
    "SD + ZLSD":"ZLSD/SD",
    "SLS+SKD":"SLS+SKD/SLS",
    "SLS":"SLS+SKD/SLS", 
    "DLGV":"DLGV/DL", 
    "DL":"DLGV/DL",
    "SMC":"SMC/GAS/Konkretno",
    "Konkretno": "SMC/GAS/Konkretno"
}

ches["Family"] = ches["Family"].map(families).fillna(ches["Family"])
ches["Parties"] = ches['Parlamint'].replace(parties).fillna(ches['Parlamint'])

ches.head()

,Parlamint,Year,lrgen,galtan,Family,Seat,Parties
0,-,2006,4.6,2.6,Green,0,-
1,-,2010,6.8,7.8,Christian-Democratic,5.6,-
2,DL,-,-,-,-,-,DLGV/DL
3,DLGV,-,-,-,-,-,DLGV/DL
4,DeSUS,2002,3.4,5.8,No family,4.4,DeSUS


## Merging ParlaMint and CHES dataset into master

In [ ]:
## Map the parties on the parliamentary speech data and extract Year from the Date column
parla["Parties"] = parla['Speaker_party'].replace(parties).fillna(parla['Speaker_party']) 
parla['Year'] = parla['Date'].str.split('-').str[0]

df = parla.merge(
    ches,
    how="left",                     # Keep all speeches
    left_on=["Parties", "Year"],    # ParlaMint side
    right_on=["Parties", "Year"]    # CHES side
)
df.head()

,ID,Text,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,...,Sents,Words,Tokens,Parties,Year,Parlamint,lrgen,galtan,Family,Seat
0,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u1,"Spoštovane poslanke, spoštovani poslanci, gosp...",2019-11-22,Term 8,Redna,Reference,Chairperson,MP,notMinister,SD,...,26,548,640,ZLSD/SD,2019,SD,2.6,2.5,Socialist,11.4
1,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u2,"Spoštovani predsednik Državnega zbora, hvala z...",2019-11-22,Term 8,Redna,Reference,Regular,notMP,notMinister,-,...,25,585,646,-,2019,NaN,NaN,NaN,NaN,NaN
2,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u3,"Hvala, gospa direktorica, za dopolnilno obrazl...",2019-11-22,Term 8,Redna,Reference,Chairperson,MP,notMinister,SD,...,3,25,31,ZLSD/SD,2019,SD,2.6,2.5,Socialist,11.4
3,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u4,"Spoštovani predsedujoči, kolegice in kolegi! O...",2019-11-22,Term 8,Redna,Reference,Regular,MP,notMinister,LMŠ,...,18,431,467,LMŠ,2019,LMŠ,4.4,4.6,Liberal,14.8
4,ParlaMint-SI_2019-11-22-SDZ8-Redna-13.u5,"Hvala, gospod predsednik, za predstavitev stal...",2019-11-22,Term 8,Redna,Reference,Chairperson,MP,notMinister,SD,...,4,33,44,ZLSD/SD,2019,SD,2.6,2.5,Socialist,11.4


In [11]:
## Check if merging was successfull (check if 2020 data is Nan, as we only have data for 2019)
ches[ches["Parties"] == "ZL/Levica"] 
parla[parla["Parties"] == "ZL/Levica"]
check = df[df["Parties"] == "ZL/Levica"]
check.head()

,ID,Text,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,...,Sents,Words,Tokens,Parties,Year,Parlamint,lrgen,galtan,Family,Seat
72,ParlaMint-SI_2019-11-22-SDZ8-Izredna-28.u10,"Hvala lepa, predsedujoči. Kolegice in kolegi! ...",2019-11-22,Term 8,Izredna,Reference,Regular,MP,notMinister,Levica,...,66,1328,1518,ZL/Levica,2019,Levica,0.7,1,Socialist,10.2
139,ParlaMint-SI_2019-11-22-SDZ8-Izredna-28.u77,"Ne, postopkovno, če lahko. Samo prosim, da naj...",2019-11-22,Term 8,Izredna,Reference,Regular,MP,notMinister,Levica,...,2,16,23,ZL/Levica,2019,Levica,0.7,1,Socialist,10.2
156,ParlaMint-SI_2019-11-26-SDZ8-Redna-13.u16,"Spoštovane, spoštovani! Po podatkih OECD smo v...",2019-11-26,Term 8,Redna,Reference,Regular,MP,notMinister,Levica,...,31,600,691,ZL/Levica,2019,Levica,0.7,1,Socialist,10.2
176,ParlaMint-SI_2019-11-26-SDZ8-Redna-13.u36,"Spoštovane, spoštovani! Danes se ponovno odloč...",2019-11-26,Term 8,Redna,Reference,Regular,MP,notMinister,Levica,...,24,419,482,ZL/Levica,2019,Levica,0.7,1,Socialist,10.2
188,ParlaMint-SI_2019-11-26-SDZ8-Redna-13.u48,"Najlepša hvala, spoštovani predsednik! Kolegic...",2019-11-26,Term 8,Redna,Reference,Regular,MP,notMinister,Levica,...,25,603,688,ZL/Levica,2019,Levica,0.7,1,Socialist,10.2


In [12]:
#Dataset saved as CSV/TSV
df.to_csv("../Sample/Datasets/Sample_master.tsv", sep='\t', encoding='utf-8', index=False)